# Visualization

**Objective:** Generate publication-quality visualizations for thesis Chapter 4 and presentations.

**Research Traceability:**
- Research Objective: Visual analysis of model performance and detection patterns
- Methodology: Matplotlib-based plots, GradCAM heatmaps, ROC curves
- Implementation: notebooks/07_visualization.ipynb

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['axes.titlesize'] = 16
%matplotlib inline

## 1. Training Curves

In [ ]:
def plot_training_curves(history_path: str, model_name: str, save: bool = True):
    """Plot training and validation loss/accuracy curves."""
    import json
    
    with open(history_path) as f:
        history = json.load(f)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Loss
    epochs = range(1, len(history['train_loss']) + 1)
    axes[0].plot(epochs, history['train_loss'], 'b-o', label='Train', markersize=4)
    axes[0].plot(epochs, history['val_loss'], 'r-o', label='Validation', markersize=4)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title(f'{model_name} - Training Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Accuracy
    axes[1].plot(epochs, history['train_acc'], 'b-o', label='Train', markersize=4)
    axes[1].plot(epochs, history['val_acc'], 'r-o', label='Validation', markersize=4)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_title(f'{model_name} - Training Accuracy')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    if save:
        plt.savefig(f'../outputs/plots/{model_name.lower()}_training.png', dpi=150, bbox_inches='tight')
    plt.show()

## 2. ROC Curve Comparison

In [ ]:
from sklearn.metrics import roc_curve, auc

def plot_roc_comparison(models_roc_data: dict, save: bool = True):
    """Plot ROC curves for multiple models on same figure."""
    plt.figure(figsize=(8, 8))
    
    colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']
    
    for (name, data), color in zip(models_roc_data.items(), colors):
        fpr, tpr, _ = roc_curve(data['y_true'], data['y_scores'])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, color=color, lw=2,
                 label=f'{name} (AUC = {roc_auc:.4f})')
    
    plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Guess')
    plt.xlim([-0.02, 1.02])
    plt.ylim([-0.02, 1.02])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve Comparison')
    plt.legend(loc='lower right')
    plt.grid(True, alpha=0.3)
    
    if save:
        plt.savefig('../outputs/plots/roc_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

## 3. Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix

def plot_confusion_matrix(y_true: np.ndarray, y_pred: np.ndarray,
                          title: str = 'Confusion Matrix',
                          save: bool = True):
    """Plot styled confusion matrix."""
    cm = confusion_matrix(y_true, y_pred)
    
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Real', 'Fake'],
                yticklabels=['Real', 'Fake'],
                ax=ax, cbar_kws={'label': 'Count'})
    
    # Add percentages
    total = cm.sum()
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j + 0.5, i + 0.7, f'({cm[i, j]/total:.1%})',
                    ha='center', va='center', fontsize=10, color='gray')
    
    ax.set_title(title, fontsize=16, pad=15)
    ax.set_xlabel('Predicted Label', fontsize=13)
    ax.set_ylabel('True Label', fontsize=13)
    
    plt.tight_layout()
    if save:
        plt.savefig('../outputs/plots/confusion_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()

## 4. Per-Manipulation Method Accuracy

In [ ]:
def plot_method_accuracy(method_results: dict, save: bool = True):
    """Plot detection accuracy per manipulation method."""
    methods = list(method_results.keys())
    accuracies = [method_results[m]['accuracy'] for m in methods]
    
    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(methods, accuracies, color=sns.color_palette('husl', len(methods)))
    
    # Add value labels
    for bar, acc in zip(bars, accuracies):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{acc:.2%}', ha='center', va='bottom', fontweight='bold')
    
    ax.set_ylabel('Accuracy')
    ax.set_title('Detection Accuracy by Manipulation Method')
    ax.set_ylim(0, 1.1)
    ax.axhline(y=0.5, color='r', linestyle='--', alpha=0.5, label='Random Baseline')
    ax.legend()
    
    plt.tight_layout()
    if save:
        plt.savefig('../outputs/plots/method_accuracy.png', dpi=150, bbox_inches='tight')
    plt.show()

## 5. Model Comparison Bar Chart

In [ ]:
import pandas as pd

def plot_model_comparison(metrics_df: pd.DataFrame, save: bool = True):
    """Plot grouped bar chart comparing models across metrics."""
    fig, ax = plt.subplots(figsize=(12, 6))
    
    metrics_df.set_index('model').plot(kind='bar', ax=ax, width=0.8)
    
    ax.set_ylabel('Score')
    ax.set_title('Model Performance Comparison')
    ax.set_ylim(0, 1.05)
    ax.legend(loc='lower right', bbox_to_anchor=(1.0, 0.0))
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    
    plt.tight_layout()
    if save:
        plt.savefig('../outputs/plots/model_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

## Summary
- Training curve visualization
- ROC curve comparison
- Confusion matrix with percentages
- Per-method accuracy breakdown
- Model comparison bar chart

All plots are saved to `outputs/plots/` for thesis inclusion.